In [1]:
print("dd")

dd


# 🦅 E-gle Eye - 실시간 관제 대시보드
**파일명:** `02_real_time_event_dashboard.ipynb`  
**위치:** `building_information` 폴더  
**실행 방법:** 아래 셀을 순서대로 실행 → 마지막 셀에서 Streamlit 앱 실행

---
> ⚠️ **사전 설치 필요:**
> ```bash
> pip install streamlit folium streamlit-folium openpyxl pandas
> ```

In [2]:
pip install streamlit folium streamlit-folium openpyxl pandas

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 24.3.1 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


## 📦 셀 1 - 라이브러리 설치 확인

In [6]:
# 필요한 패키지 설치 (최초 1회)
import subprocess
import sys

packages = ["streamlit", "folium", "streamlit-folium", "openpyxl", "pandas"]

for pkg in packages:
    result = subprocess.run(
        [sys.executable, "-m", "pip", "install", pkg, "--quiet"],
        capture_output=True, text=True
    )
    status = "✅" if result.returncode == 0 else "❌"
    print(f"{status} {pkg}")

print("\n모든 패키지 확인 완료!")

✅ streamlit
✅ folium
✅ streamlit-folium
✅ openpyxl
✅ pandas

모든 패키지 확인 완료!


## 📁 셀 2 - 프로젝트 경로 설정 및 확인

In [11]:
import sys
import os

# =====================================================
# ✏️ 프로젝트 경로를 본인 환경에 맞게 수정하세요
# =====================================================
PROJECT_DIR = r"E:\newfolder\Egle_project\AzureMachineLearning\building_information"

# 경로가 존재하는지 확인
if os.path.exists(PROJECT_DIR):
    print(f"✅ 프로젝트 경로 확인: {PROJECT_DIR}")
else:
    print(f"❌ 경로 없음: {PROJECT_DIR}")
    print("→ PROJECT_DIR 변수를 실제 경로로 수정해주세요.")

# sys.path 등록
if PROJECT_DIR not in sys.path:
    sys.path.insert(0, PROJECT_DIR)
    print(f"✅ sys.path에 추가됨")
else:
    print(f"ℹ️  이미 sys.path에 등록되어 있음")

# building_location_manager.py 존재 여부 확인
manager_path = os.path.join(PROJECT_DIR, "building_location_manager.py")
if os.path.exists(manager_path):
    print(f"✅ building_location_manager.py 발견")
else:
    print(f"⚠️  building_location_manager.py 미발견 → 아래 셀에서 Mock 모드 사용")

✅ 프로젝트 경로 확인: E:\newfolder\Egle_project\AzureMachineLearning\building_information
ℹ️  이미 sys.path에 등록되어 있음
✅ building_location_manager.py 발견


## 🏢 셀 3 - BuildingLocationManager 로드 및 카메라 데이터 확인

In [19]:
import pandas as pd

# =====================================================
# BuildingLocationManager 로드 시도
# 실패 시 Mock 데이터로 대체 (개발/테스트용)
# =====================================================

USE_MOCK = False  # 실제 매니저 사용 여부

try:
    from building_location_manager import BuildingLocationManager
    blm = BuildingLocationManager()
    print("✅ BuildingLocationManager 로드 성공")
except ImportError as e:
    print(f"⚠️  ImportError: {e}")
    print("→ Mock 데이터 모드로 전환합니다.")
    USE_MOCK = True
except Exception as e:
    print(f"⚠️  Error: {e}")
    print("→ Mock 데이터 모드로 전환합니다.")
    USE_MOCK = True

# Mock 데이터 (실제 매니저 없을 때 사용)
MOCK_CAMERAS = [
    {"Camera_ID": "camera_01", "Building_Name": "A동 창고",   "Zone": "Zone-A", "GPS_Lat": 37.4790, "GPS_Lon": 126.6310, "status": "Green",  "Fire_Dept_Phone": "032-111-1111"},
    {"Camera_ID": "camera_02", "Building_Name": "B동 창고",   "Zone": "Zone-B", "GPS_Lat": 37.4785, "GPS_Lon": 126.6320, "status": "Green",  "Fire_Dept_Phone": "032-111-1111"},
    {"Camera_ID": "camera_03", "Building_Name": "C동 사무동", "Zone": "Zone-C", "GPS_Lat": 37.4780, "GPS_Lon": 126.6330, "status": "Orange", "Fire_Dept_Phone": "032-222-2222"},
    {"Camera_ID": "camera_04", "Building_Name": "D동 물류",   "Zone": "Zone-D", "GPS_Lat": 37.4775, "GPS_Lon": 126.6340, "status": "Orange", "Fire_Dept_Phone": "032-222-2222"},
    {"Camera_ID": "camera_05", "Building_Name": "E동 하역장", "Zone": "Zone-E", "GPS_Lat": 37.4795, "GPS_Lon": 126.6350, "status": "Red",    "Fire_Dept_Phone": "032-333-3333"},
    {"Camera_ID": "camera_06", "Building_Name": "F동 변전소", "Zone": "Zone-F", "GPS_Lat": 37.4800, "GPS_Lon": 126.6325, "status": "Green",  "Fire_Dept_Phone": "032-333-3333"},
    {"Camera_ID": "camera_07", "Building_Name": "G동 주차장", "Zone": "Zone-G", "GPS_Lat": 37.4770, "GPS_Lon": 126.6315, "status": "Green",  "Fire_Dept_Phone": "032-444-4444"},
    {"Camera_ID": "camera_08", "Building_Name": "H동 게이트", "Zone": "Zone-H", "GPS_Lat": 37.4765, "GPS_Lon": 126.6360, "status": "Green",  "Fire_Dept_Phone": "032-444-4444"},
]

# 카메라 데이터 수집
cams = []
if USE_MOCK:
    cams = MOCK_CAMERAS
    print(f"\n📋 Mock 카메라 {len(cams)}대 로드됨")
else:
    for i in range(1, 9):
        camera_id = f"camera_{i:02d}"
        info = blm.get_full_camera_info(camera_id)
        if info:
            info["status"] = info.get("status", "Green")
            cams.append(info)
    print(f"\n📋 실제 카메라 {len(cams)}대 로드됨")

#데이터 미리보기
df_preview = pd.DataFrame(cams)[["Camera_ID", "Building_Name", "Zone", "status", "GPS_Lat", "GPS_Lon"]]
print("\n📊 카메라 현황:")
display(df_preview)

📂 기존 파일 로드: buildings2.xlsx
✅ BuildingLocationManager 초기화 완료 (8개 카메라 등록됨)
✅ BuildingLocationManager 로드 성공

📋 실제 카메라 8대 로드됨

📊 카메라 현황:


,Camera_ID,Building_Name,Zone,status,GPS_Lat,GPS_Lon
0,camera_01,인천 북항 스마트 물류센터,입고 게이트 A,Green,37.4782,126.6321
1,camera_02,인천 북항 스마트 물류센터,입고 게이트 B,Green,37.4785,126.6324
2,camera_03,인천 북항 스마트 물류센터,생산라인 1층,Green,37.4791,126.6318
3,camera_04,인천 북항 스마트 물류센터,생산라인 2층,Green,37.4793,126.6320
4,camera_05,인천 북항 스마트 물류센터,출고 적재장,Green,37.4778,126.6335
5,camera_06,인천 북항 스마트 물류센터,사무동 3층,Green,37.4802,126.6312
6,camera_07,인천 북항 스마트 물류센터,야외 보안 주차장,Green,37.4765,126.6341
7,camera_08,인천 북항 스마트 물류센터,중앙 보안센터,Green,37.4789,126.6327


## 🗺️ 셀 4 - Folium 지도 미리보기 (Jupyter 인라인)

In [14]:
# =====================================================
# E-gle Eye - 실시간 카메라 위치 지도 (Folium)
# [추가됨 ★★★★★] BuildingLocationManager 완전 연동 버전
# 원준님 ipynb 전용 / 2026-03-04
# =====================================================

import folium
from building_location_manager import BuildingLocationManager   # ← 기존 파일 import

# [추가됨] 매니저 로드 (8대 카메라 자동 등록)
manager = BuildingLocationManager()

# [추가됨] 모든 카메라 정보 가져오기 (status 포함)
cams = []
for cam_id in manager.df["Camera_ID"].tolist():
    full_info = manager.get_full_camera_info(cam_id)
    if full_info:
        cams.append(full_info)

# 상태별 색상/이모지 (기존 코드 유지 + 보강)
STATUS_COLOR = {"Green": "green", "Orange": "orange", "Red": "red"}
STATUS_EMOJI = {"Green": "🟢", "Orange": "🟠", "Red": "🔴"}

# 지도 생성 (인천 북항 스마트 물류센터 중심)
m = folium.Map(
    location=[37.4785, 126.6325],
    zoom_start=15,
    tiles="cartodb positron"
)

# [추가됨] 8대 카메라 마커 자동 생성
for cam in cams:
    color = STATUS_COLOR.get(cam.get("status"), "gray")
    emoji = STATUS_EMOJI.get(cam.get("status"), "⚪")

    popup_html = f"""
    <div style="font-family: Arial; min-width:200px; padding:8px;">
        <h4 style="margin:0; color:#333;">📷 {cam['Camera_ID']}</h4>
        <hr style="margin:6px 0;">
        <p style="margin:3px 0;">🏢 <b>{cam.get('Building_Name', 'N/A')}</b></p>
        <p style="margin:3px 0;">📍 Zone: {cam.get('Zone', 'N/A')}</p>
        <p style="margin:3px 0;">📊 상태: <b>{emoji} {cam.get('status', 'N/A')}</b></p>
        <p style="margin:3px 0;">🚨 소방서: {cam.get('Fire_Dept_Phone', 'N/A')}</p>
        <p style="margin:3px 0;">📍 GPS: {cam.get('GPS_Lat'):.4f}, {cam.get('GPS_Lon'):.4f}</p>
        <hr style="margin:6px 0;">
        <small style="color:#0066cc;">🎥 최근 이벤트: {len(cam.get('recent_events', []))}건</small>
    </div>
    """

    folium.Marker(
        location=[cam.get("GPS_Lat"), cam.get("GPS_Lon")],
        popup=folium.Popup(popup_html, max_width=250),
        tooltip=f"{cam['Camera_ID']} | {emoji} {cam.get('status')}",
        icon=folium.Icon(color=color, icon="video", prefix="fa")
    ).add_to(m)

print(f"✅ E-gle Eye 실시간 지도 생성 완료 ({len(cams)}대 카메라 표시)")
print("   → Trust Notebook 실행 후 아래에서 인터랙티브 지도 확인하세요!")

# Jupyter에서 바로 표시


📂 기존 파일 로드: buildings2.xlsx
✅ BuildingLocationManager 초기화 완료 (3개 카메라 등록됨)
✅ E-gle Eye 실시간 지도 생성 완료 (3대 카메라 표시)
   → Trust Notebook 실행 후 아래에서 인터랙티브 지도 확인하세요!


## 📊 셀 5 - 카메라 상태 요약 통계

In [15]:
from collections import Counter

status_counts = Counter(cam.get("status", "Unknown") for cam in cams)

print("=" * 40)
print("       📊 카메라 상태 요약")
print("=" * 40)
print(f"  🟢 Green  (정상)   : {status_counts.get('Green',  0)}대")
print(f"  🟠 Orange (주의)   : {status_counts.get('Orange', 0)}대")
print(f"  🔴 Red    (위험)   : {status_counts.get('Red',    0)}대")
print("-" * 40)
print(f"     합계              : {len(cams)}대")
print("=" * 40)

# 경보 카메라 강조
alert_cams = [c for c in cams if c.get("status") in ("Orange", "Red")]
if alert_cams:
    print("\n⚠️  주의/위험 카메라 목록:")
    for ac in alert_cams:
        emoji = STATUS_EMOJI.get(ac.get("status"), "⚪")
        print(f"   {emoji} {ac['Camera_ID']} | {ac.get('Building_Name')} | {ac.get('Zone')}")
else:
    print("\n✅ 모든 카메라 정상 상태입니다.")

       📊 카메라 상태 요약
  🟢 Green  (정상)   : 3대
  🟠 Orange (주의)   : 0대
  🔴 Red    (위험)   : 0대
----------------------------------------
     합계              : 3대

✅ 모든 카메라 정상 상태입니다.


## 📝 셀 6 - Streamlit 앱 코드 파일 생성

In [ ]:
# =====================================================
# Streamlit 앱 소스 코드를 .py 파일로 저장
# → 저장 후 터미널에서 실행
# =====================================================

streamlit_code = '''
# =====================================================
# 02_real_time_event_dashboard.py  (자동 생성)
# E-gle Eye - Real-time Event Dashboard + 가상 지도
# 실행: streamlit run 02_real_time_event_dashboard.py
# =====================================================

import sys
import os
import streamlit as st
import folium
from streamlit_folium import st_folium

# =====================================================
# ✏️ 프로젝트 경로 (본인 환경에 맞게 수정)
# =====================================================
PROJECT_DIR = r"E:\\newfolder\\Egle_project\\AzureMachineLearning\\building_information"
sys.path.insert(0, PROJECT_DIR)

# ----------------------------------
# 상태 색상/이모지 매핑
# ----------------------------------
STATUS_COLOR = {"Green": "green", "Orange": "orange", "Red": "red"}
STATUS_EMOJI = {"Green": "🟢", "Orange": "🟠", "Red": "🔴"}
STATUS_BG    = {"Green": "#d4edda", "Orange": "#fff3cd", "Red": "#f8d7da"}

# ----------------------------------
# Mock 카메라 데이터 (매니저 없을 때)
# ----------------------------------
MOCK_CAMERAS = [
    {"Camera_ID": "camera_01", "Building_Name": "A동 창고",   "Zone": "Zone-A", "GPS_Lat": 37.4790, "GPS_Lon": 126.6310, "status": "Green",  "Fire_Dept_Phone": "032-111-1111"},
    {"Camera_ID": "camera_02", "Building_Name": "B동 창고",   "Zone": "Zone-B", "GPS_Lat": 37.4785, "GPS_Lon": 126.6320, "status": "Green",  "Fire_Dept_Phone": "032-111-1111"},
    {"Camera_ID": "camera_03", "Building_Name": "C동 사무동", "Zone": "Zone-C", "GPS_Lat": 37.4780, "GPS_Lon": 126.6330, "status": "Orange", "Fire_Dept_Phone": "032-222-2222"},
    {"Camera_ID": "camera_04", "Building_Name": "D동 물류",   "Zone": "Zone-D", "GPS_Lat": 37.4775, "GPS_Lon": 126.6340, "status": "Orange", "Fire_Dept_Phone": "032-222-2222"},
    {"Camera_ID": "camera_05", "Building_Name": "E동 하역장", "Zone": "Zone-E", "GPS_Lat": 37.4795, "GPS_Lon": 126.6350, "status": "Red",    "Fire_Dept_Phone": "032-333-3333"},
    {"Camera_ID": "camera_06", "Building_Name": "F동 변전소", "Zone": "Zone-F", "GPS_Lat": 37.4800, "GPS_Lon": 126.6325, "status": "Green",  "Fire_Dept_Phone": "032-333-3333"},
    {"Camera_ID": "camera_07", "Building_Name": "G동 주차장", "Zone": "Zone-G", "GPS_Lat": 37.4770, "GPS_Lon": 126.6315, "status": "Green",  "Fire_Dept_Phone": "032-444-4444"},
    {"Camera_ID": "camera_08", "Building_Name": "H동 게이트", "Zone": "Zone-H", "GPS_Lat": 37.4765, "GPS_Lon": 126.6360, "status": "Green",  "Fire_Dept_Phone": "032-444-4444"},
]

# ----------------------------------
# 페이지 설정
# ----------------------------------
st.set_page_config(page_title="E-gle Eye 관제", layout="wide", page_icon="🦅")

# CSS 커스텀
st.markdown("""
<style>
  .alert-card { padding: 10px 14px; border-radius: 8px; margin-bottom: 6px; border-left: 5px solid; }
  .alert-red    { background:#f8d7da; border-color:#dc3545; }
  .alert-orange { background:#fff3cd; border-color:#fd7e14; }
  .alert-green  { background:#d4edda; border-color:#28a745; }
  .section-title { font-size:1.15rem; font-weight:700; margin-bottom:8px; }
</style>
""", unsafe_allow_html=True)

st.title("🦅 E-gle Eye — 실시간 관제 대시보드")
st.caption("인천 북항 스마트 물류센터 | 신뢰도 기반 화재 감지 시스템")

# ----------------------------------
# 사이드바 필터
# ----------------------------------
st.sidebar.header("🔍 필터")
status_filter = st.sidebar.multiselect(
    "상태 선택",
    ["Green", "Orange", "Red"],
    default=["Green", "Orange", "Red"]
)
st.sidebar.markdown("---")
st.sidebar.info("💡 마커를 클릭하면 카메라 상세 정보를 확인할 수 있습니다.")

# ----------------------------------
# 카메라 데이터 로드
# ----------------------------------
cams = []
try:
    from building_location_manager import BuildingLocationManager
    blm = BuildingLocationManager()
    for i in range(1, 9):
        camera_id = f"camera_{i:02d}"
        info = blm.get_full_camera_info(camera_id)
        if info:
            info["status"] = info.get("status", "Green")
            cams.append(info)
    st.sidebar.success(f"✅ 실제 데이터 {len(cams)}대 로드")
except Exception:
    cams = MOCK_CAMERAS
    st.sidebar.warning("⚠️ Mock 데이터 사용 중")

# 필터 적용
filtered_cams = [c for c in cams if c.get("status") in status_filter]

# ----------------------------------
# 상단 요약 KPI
# ----------------------------------
from collections import Counter
cnt = Counter(c.get("status") for c in cams)

k1, k2, k3, k4 = st.columns(4)
k1.metric("📷 전체 카메라",  f"{len(cams)}대")
k2.metric("🟢 Green  정상",  f"{cnt.get('Green',  0)}대")
k3.metric("🟠 Orange 주의",  f"{cnt.get('Orange', 0)}대",
          delta="주의" if cnt.get('Orange', 0) > 0 else None,
          delta_color="inverse")
k4.metric("🔴 Red    위험",  f"{cnt.get('Red',    0)}대",
          delta="긴급" if cnt.get('Red', 0) > 0 else None,
          delta_color="inverse")

st.markdown("---")

# ----------------------------------
# 지도 + 경보 목록 레이아웃
# ----------------------------------
col_map, col_alert = st.columns([3, 1])

with col_map:
    st.subheader("🗺️ 실시간 가상 지도")

    # Folium 지도 생성
    m = folium.Map(
        location=[37.4785, 126.6325],
        zoom_start=15,
        tiles="cartodb positron"
    )

    for cam in filtered_cams:
        color = STATUS_COLOR.get(cam.get("status"), "gray")
        emoji = STATUS_EMOJI.get(cam.get("status"), "⚪")

        popup_html = f"""
        <div style="font-family:Arial; min-width:190px;">
            <h4 style="margin:0 0 4px; color:#333;">📷 {cam[\'Camera_ID\']}</h4>
            <hr style="margin:4px 0;">
            <p style="margin:2px 0;">🏢 <b>{cam.get(\'Building_Name\', \'-\')}</b></p>
            <p style="margin:2px 0;">📍 Zone: {cam.get(\'Zone\', \'-\')}</p>
            <p style="margin:2px 0;">📊 상태: <b>{emoji} {cam.get(\'status\', \'-\')}</b></p>
            <p style="margin:2px 0;">🚨 소방서: {cam.get(\'Fire_Dept_Phone\', \'-\')}</p>
            <hr style="margin:4px 0;">
            <small style="color:#888;">🎥 클립 보기 (추후 연동)</small>
        </div>
        """
        folium.Marker(
            location=[cam.get("GPS_Lat"), cam.get("GPS_Lon")],
            popup=folium.Popup(popup_html, max_width=220),
            tooltip=f"{cam[\'Camera_ID\']} | {cam.get(\'status\')}",
            icon=folium.Icon(color=color, icon="fire", prefix="fa")
        ).add_to(m)

    st_folium(m, width=950, height=500)

with col_alert:
    st.subheader("🚨 경보 목록")

    red_list    = [c for c in cams if c.get("status") == "Red"]
    orange_list = [c for c in cams if c.get("status") == "Orange"]

    if red_list:
        st.markdown("**🔴 위험 (RED)**")
        for c in red_list:
            st.markdown(
                f\'<div class="alert-card alert-red">\'
                f\'<b>{c["Camera_ID"]}</b><br>\'
                f\'{c.get("Building_Name")} | {c.get("Zone")}</div>\',
                unsafe_allow_html=True
            )

    if orange_list:
        st.markdown("**🟠 주의 (ORANGE)**")
        for c in orange_list:
            st.markdown(
                f\'<div class="alert-card alert-orange">\'
                f\'<b>{c["Camera_ID"]}</b><br>\'
                f\'{c.get("Building_Name")} | {c.get("Zone")}</div>\',
                unsafe_allow_html=True
            )

    if not red_list and not orange_list:
        st.success("모든 카메라 정상")

st.markdown("---")

# ----------------------------------
# 8대 카메라 상태 카드
# ----------------------------------
st.subheader("📊 8대 카메라 실시간 상태")
cols = st.columns(4)
for i, cam in enumerate(cams):
    with cols[i % 4]:
        status = cam.get("status", "Green")
        emoji  = STATUS_EMOJI.get(status, "⚪")
        bg     = STATUS_BG.get(status, "#f0f0f0")
        st.markdown(
            f\'<div style="background:{bg}; padding:12px; border-radius:8px; margin-bottom:8px;">\'
            f\'<b>{emoji} {cam["Camera_ID"]}</b><br>\'
            f\'{cam.get("Building_Name")}<br>\'
            f\'<small>{cam.get("Zone")} | {status}</small>\'
            f\'</div>\',
            unsafe_allow_html=True
        )

st.success("✅ E-gle Eye 대시보드 정상 작동 중!")
'''

# .py 파일로 저장 (notebook과 같은 폴더)
output_path = os.path.join(PROJECT_DIR, "02_real_time_event_dashboard.py")

# PROJECT_DIR이 없으면 현재 디렉터리에 저장
if not os.path.exists(PROJECT_DIR):
    output_path = os.path.join(os.getcwd(), "02_real_time_event_dashboard.py")

with open(output_path, "w", encoding="utf-8") as f:
    f.write(streamlit_code.strip())

print(f"✅ Streamlit 앱 파일 저장 완료!")
print(f"📁 저장 위치: {output_path}")
print(f"\n▶️  터미널에서 아래 명령어로 실행하세요:")
print(f"   streamlit run \"{output_path}\"")

## 🚀 셀 7 - Streamlit 앱 실행

> ⚠️ **Jupyter에서는 직접 실행이 어렵습니다.**  
> 아래 셀을 실행하면 **VSCode 터미널에서 실행할 명령어**를 출력합니다.
> 또는 `!` 명령으로 백그라운드 실행을 시도할 수 있습니다.

In [ ]:
import os

# PROJECT_DIR이 없으면 현재 폴더 사용
if not os.path.exists(PROJECT_DIR):
    run_dir = os.getcwd()
else:
    run_dir = PROJECT_DIR

app_path = os.path.join(run_dir, "02_real_time_event_dashboard.py")

print("=" * 60)
print("  🚀 Streamlit 실행 방법")
print("=" * 60)
print("\n방법 1️⃣  VSCode 터미널에서 직접 실행 (권장)")
print(f"  streamlit run \"{app_path}\"")
print("\n방법 2️⃣  아래 셀의 주석 해제 후 실행 (백그라운드)")
print("  # !streamlit run \"파일경로\" &")
print("\n방법 3️⃣  브라우저에서 접속")
print("  http://localhost:8501")
print("=" * 60)

In [ ]:
# =====================================================
# ▶️  아래 주석을 해제하면 Jupyter에서 직접 실행 가능
#    (단, 셀이 멈춰있는 것처럼 보이는 건 정상)
# =====================================================

# import subprocess, sys, os

# if not os.path.exists(PROJECT_DIR):
#     run_dir = os.getcwd()
# else:
#     run_dir = PROJECT_DIR

# app_path = os.path.join(run_dir, "02_real_time_event_dashboard.py")

# proc = subprocess.Popen(
#     [sys.executable, "-m", "streamlit", "run", app_path],
#     stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True
# )
# print("🚀 Streamlit 시작됨 → http://localhost:8501")
# print("종료하려면 커널을 재시작하거나 Ctrl+C를 누르세요.")
# for line in proc.stdout:
#     print(line, end="")

---
## 📋 사용 가이드

| 셀 | 역할 |
|---|---|
| 셀 1 | 패키지 설치 확인 |
| 셀 2 | 프로젝트 경로 설정 → **PROJECT_DIR 수정 필요** |
| 셀 3 | BuildingLocationManager 로드 (실패 시 Mock 데이터 사용) |
| 셀 4 | Jupyter 인라인 Folium 지도 미리보기 |
| 셀 5 | 카메라 상태 통계 출력 |
| 셀 6 | Streamlit `.py` 파일 자동 생성 |
| 셀 7 | 실행 명령어 출력 + 백그라운드 실행 옵션 |

### 🔧 건물 상태 변경 방법
셀 3의 `MOCK_CAMERAS` 리스트에서 `"status"` 값을 `"Green"` / `"Orange"` / `"Red"` 로 수정하세요.

### ✅ 실제 데이터 연결
`셀 2`의 `PROJECT_DIR` 경로가 올바르고 `building_location_manager.py`가 존재하면 자동으로 실제 데이터를 사용합니다.